# Clase 188 — Inferencia causal: DAGs, confounders, instrumentos

Correlación ≠ causalidad. Simulamos un DAG con numpy y mostramos: el sesgo por **confounder** omitido y su corrección por ajuste, el daño de controlar un **collider**, y la recuperación del efecto con **variables instrumentales (2SLS)** cuando hay confounders no observados.

Requiere: `numpy`, `pandas`, `statsmodels`, `matplotlib`.

## 1. Confounder (fork): sesgo por variable omitida

`T ← Z → Y`. `Z` causa tanto `T` como `Y`. El OLS ingenuo `Y~T` está sesgado; ajustar por `Z` recupera el efecto causal (backdoor criterion).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.sandbox.regression.gmm import IV2SLS
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 5000
Z = rng.normal(0, 1, n)                     # confounder
T = 0.8 * Z + rng.normal(0, 1, n)           # tratamiento depende de Z
Y = 2.0 * T + 3.0 * Z + rng.normal(0, 1, n) # efecto causal verdadero de T = 2.0
df = pd.DataFrame({"Y": Y, "T": T, "Z": Z})
naive = smf.ols("Y ~ T", data=df).fit().params["T"]
adj   = smf.ols("Y ~ T + Z", data=df).fit().params["T"]
print("efecto verdadero = 2.00")
print(f"OLS ingenuo   Y~T   = {naive:.3f}  (sesgado por Z)")
print(f"OLS ajustado  Y~T+Z = {adj:.3f}  (recupera el efecto)")
assert abs(adj - 2.0) < 0.15 and abs(naive - 2.0) > 0.3

## 2. Collider: controlar de más introduce sesgo

`T → C ← Y`. Controlar el collider `C` **crea** una asociación espuria y destruye la estimación. Controlar "todo lo que tengo" es un error.

In [ ]:
T2 = rng.normal(0, 1, n)
Y2 = rng.normal(T2, 1)                 # Y depende de T (efecto 1.0)
C = T2 + Y2 + rng.normal(0, 1, n)      # collider: hijo de T y de Y
d2 = pd.DataFrame({"Y": Y2, "T": T2, "C": C})
b_free = smf.ols("Y ~ T", data=d2).fit().params["T"]
b_ctrl = smf.ols("Y ~ T + C", data=d2).fit().params["T"]
print(f"sin controlar C   Y~T   = {b_free:.3f}  (≈1, correcto)")
print(f"controlando C     Y~T+C = {b_ctrl:.3f}  (sesgado: controlar el collider DAÑA)")
assert abs(b_free - 1.0) < 0.1 and b_ctrl < b_free

## 3. Variables instrumentales (2SLS)

Con un confounder **no observado** `U` entre `T` e `Y`, el OLS es inconsistente. Un instrumento `Z` (afecta a `T`, no a `Y` salvo vía `T`) identifica el efecto con 2SLS.

In [ ]:
U = rng.normal(0, 1, n)                          # confounder NO observado
Zi = rng.normal(0, 1, n)                         # instrumento
Ti = 0.7 * Zi + 1.0 * U + rng.normal(0, 1, n)
Yi = 1.5 * Ti + 2.0 * U + rng.normal(0, 1, n)    # efecto verdadero 1.5
dta = pd.DataFrame({"Y": Yi, "T": Ti, "Z": Zi})

ols_biased = smf.ols("Y ~ T", data=dta).fit().params["T"]
exog  = sm.add_constant(dta[["T"]])
instr = sm.add_constant(dta[["Z"]])
iv = IV2SLS(dta["Y"], exog, instrument=instr).fit()
first = smf.ols("T ~ Z", data=dta).fit()
print("efecto verdadero = 1.50")
print(f"OLS (U no observado) = {ols_biased:.3f}  (sesgado)")
print(f"IV 2SLS              = {iv.params['T']:.3f}  (recupera ~1.5)")
print(f"F 1ª etapa = {first.fvalue:.1f}  (>10 => instrumento fuerte, regla de Stock-Yogo)")
assert abs(iv.params["T"] - 1.5) < 0.2 and first.fvalue > 10

## 4. Resumen visual del sesgo

Comparamos cada estimador contra su valor verdadero.

In [ ]:
labels = ["ingenuo\n(confounder)", "ajustado\n(backdoor)", "OLS\n(U no obs)", "IV 2SLS"]
vals   = [naive, adj, ols_biased, iv.params["T"]]
truths = [2.0, 2.0, 1.5, 1.5]
xpos = np.arange(len(labels))
plt.figure(figsize=(7, 4))
plt.bar(xpos, vals, color="steelblue", label="estimado")
plt.plot(xpos, truths, "rD", ms=11, label="verdadero")
plt.xticks(xpos, labels); plt.legend()
plt.title("Sesgo por confounders y su corrección")
plt.tight_layout(); plt.show()

## Ejercicios

1. Repetí el bloque 1 pero controlando además una variable irrelevante `W ~ N(0,1)`: verificá que no cambia el efecto estimado.
2. Debilitá el instrumento (`Ti = 0.05·Zi + U + ε`) y observá cómo la F de la 1ª etapa cae debajo de 10 y el 2SLS se vuelve inestable.
3. Simulá un mediator `T → M → Y` y mostrá que controlar `M` sesga el efecto total hacia 0.

## Conclusiones

- Dibujá el DAG **antes** de decidir qué controlar: no todo control mejora la estimación.
- Confounder (fork) → ajustar; collider → NO ajustar; mediator → ajustar bloquea el efecto indirecto.
- El OLS es causal solo bajo unconfoundedness; con confounders no observados usá IV, DiD o RDD.
- Un instrumento débil (F < 10) produce estimaciones sesgadas e inestables.